## Part 2: Prepare ELSET Data
#### 1. Acquire the latest General Perturbation (gp) Data in form of Orbit Mean-Element Messages (OMMs) from [space-track.org](https://www.space-track.org/)
#### Last Updated: 2025-03-17

In [3]:
# Reauthenticate with Space-Track.org to avoid having to run first cell again

# Space-Track.org API authentication 
from spacetrack import SpaceTrackClient
import pandas as pd

# read Space-Track.org credentials from external file (username in line 1, password in line 2)
with open('../SpaceTrackCredentials.txt', 'r') as file:
    credentials = file.read().splitlines()

username = credentials[0]
password = credentials[1]

# create a SpaceTrackClient instance
st = SpaceTrackClient(username, password) 

try:
    # Try fetching a single SATCAT entry to verify authentication
    test_data = st.gp(limit=1, format='json')
    print("Successfully authenticated with Space-Track.org!")
except Exception as e:
    print(f"Failed to authenticate with Space-Track.org: {e}")

Successfully authenticated with Space-Track.org!


In [4]:
# Read the NORAD IDs from the file
norad_ids = pd.read_csv("../Data/NoradIDs.csv")
print(f"Objects: {len(norad_ids)}")

Objects: 25798


In [5]:
# Split the IDs into chunks of 500 to avoid violating the API query restrictions
norad_ids_chunks = []
chunk_size = 500
for i in range(0, len(norad_ids), chunk_size):
    chunk = norad_ids[i:i + chunk_size]
    norad_ids_chunks.append(chunk)

# show number and size of chunks
print(f"Number of chunks: {len(norad_ids_chunks)}")
print(f"Chunk size: {len(norad_ids_chunks[0])}; last chunk: {len(norad_ids_chunks[-1])}")

Number of chunks: 52
Chunk size: 500; last chunk: 298


In [6]:
# Convert each chunk to a string to avoid violating the API query limit
chunk_strings=[]
for chunk in norad_ids_chunks:
    if not chunk.empty:
        chunk_string = ",".join(chunk["NORAD_CAT_ID"].astype(str))
        chunk_strings.append(chunk_string)


##### Process chunks automatically

In [107]:
# *** imports ***
import time
import json
import os
from httpx import ReadTimeout
from spacetrack import SpaceTrackClient

# *** settings ***
output_file = "../Data/AllEpochs.csv"
initial_chunk_size = 500
request_delay = 2

# *** Functions ***
# Re-login 
def login_spacetrack():
    # read Space-Track.org credentials from external file (username in line 1, password in line 2)
    with open('../SpaceTrackCredentials.txt', 'r') as file:
        credentials = file.read().splitlines()

    username = credentials[0]
    password = credentials[1]

# function to fetch omms (epochs)
def fetch_omms(chunk): 
    try:
        api_url = f"https://www.space-track.org/basicspacedata/query/class/gp_history/NORAD_CAT_ID/{','.join(chunk)}/format/json"
        omm_data = st.gp_history(norad_cat_id=','.join(chunk), format='json')
        omm_data_json = json.loads(omm_data) if omm_data else []
        # Extract only the required fields immediately to minimize RAM usage
        extracted_data = [{'NORAD_CAT_ID': entry['NORAD_CAT_ID'], 'EPOCH': entry['EPOCH']}
                          for entry in omm_data_json]
        return extracted_data
    except ReadTimeout:
        print(f"Read Timeout occured for {len(chunk)} objects.")
        return "TIMEOUT"
    except Exception as e:
        print(f"Request failed: {e}")
        return None

# Main function to retrieve all Epochs
def retrieve_all_epochs():
    chunk_size = initial_chunk_size
    index = 0
    total_objects = len(norad_ids)
    consecutive_timeouts = 0

    while index < total_objects:
        current_chunk = norad_ids[index:index + chunk_size] 
        print(f"Querying {len(current_chunk)} objects from index {index} to {index+len(current_chunk)-1}...")
        print(f"API URL: {api_url}")

        extracted_data = fetch_omms(current_chunk)
        if extracted_data == "TIMEOUT":
            consecutive_timeouts += 1
            print(f"Timeout error ({consecutive_timeouts})")
            if consecutive_timeouts >= 2:
                new_chunk_size = chunk_size // 2
                print(f"Two consecutive timeouts. Reducing chunk size from {chunk_size} to {new_chunk_size}.")
                chunk_size = new_chunk_size
                consecutive_timeouts = 0
            continue
        elif extracted_data is None:
            print("Skipping this batch due to an error.")
            index += chunk_size
            consecutive_timeouts = 0
            continue

        consecutive_timeouts = 0

        df_chunk = pd.DataFrame(extracted_data)

        # log results
        num_ids_with_omm = df_chunk['NORAD_CAT_ID'].nunique()
        total_omms = len(df_chunk)
        print(f"Chunk {index // chunk_size}: Requested {len(current_chunk)} objects, {num_ids_with_omm} returned at least one OMM, {total_omms} OMMs in total retrieved.")

        df_chunk.to_csv(output_file, mode='a', index=False, header=not os.path.exists(output_file))
        index += chunk_size
        time.sleep(request_delay)
    
    print("Retrieval complete.")

# *** Execution ***
# load previous data if it exists
if os.path.exists(output_file):
    retrieved_epochs = pd.read_csv(output_file)
else:
    retrieved_epochs = pd.DataFrame(columns=['NORAD_CAT_ID', 'EPOCH'])

retrieve_all_epochs()

Querying 500 objects from index 0 to 499...
API URL: https://www.space-track.org/basicspacedata/query/class/gp_history/NORAD_CAT_ID/40518,45622,45909,55005,58229,12279,34292,52960,55694,55849,55903,59372,23260,29746,47187,50284,51482,60231,33622,35772,49232,60080,60219,60232,58615,14807,26178,31077,31434,32072,35253,35374,37088,39369,60218,60221,60223,60224,60226,60228,14841,32194,36143,37506,46982,52958,18643,18864,18872,19054,19454,19455,19965,20456,27730,39099,58144,4206,26416,27751,34643,45980,49789,49840,51483,60215,60230,59047,33666,36076,49480,36732,31690,31953,35581,36387,38196,39899,47064,52921,54980,31751,57279,10205,34055,34130,5842,21382,26448,30024,30333,33873,34038,34066,35278,40431,41129,43103,50146,51455,51480,52035,61044,41024,42336,21335,29102,31238,34295,30491,54961,33770,58561,5502,22473,22499,30233,31478,32158,32202,33888,35066,35858,37342,38530,41303,47060,48202,56264,60846,19006,34213,40679,41095,52030,52936,60806,28599,31174,60867,34456,42304,54399,21970,30227,5

KeyError: 'NORAD_CAT_ID'

In [7]:
import time
import json
import os
from httpx import ReadTimeout

# Retreive OMM data for each chunk
# Load previously saved data if it exists
if os.path.exists('../Data/AllEpochs.csv'):
    OMMs_df = pd.read_csv('../Data/AllEpochs.csv')
    with open('../Data/last_chunk.txt', 'r') as f:
        start_chunk = int(f.read().strip())
    omm_data_chunks = OMMs_df.to_dict('records')
else:
    OMMs_df = pd.DataFrame()
    omm_data_chunks = []
    start_chunk = 0

request_counter = 0  # Track API request count
last_chunk_processed = start_chunk
error_occurred = False  # Flag to indicate if an error occurred

for i, chunk_string in enumerate(chunk_strings[start_chunk:], start=start_chunk):
    retry_count = 0  # Initialize retry counter
    while True:
        try:
            # Construct the API URL for debugging purposes
            api_url = f"https://www.space-track.org/basicspacedata/query/class/gp_history/NORAD_CAT_ID/{chunk_string}/format/json"
            
            # retreive omm data
            omm_data = st.gp_history(norad_cat_id=chunk_string, format='json')
            request_counter += 1

            # Parse the json response and filter the required fields
            omm_data_json = json.loads(omm_data)
            filtered_omm_data = [{'NORAD_CAT_ID': entry['NORAD_CAT_ID'], 'EPOCH': entry['EPOCH']} for entry in omm_data_json]
            
            # Check if the response consists of 0 elements
            if len(filtered_omm_data) == 0:
                raise Exception(f"No data retrieved for chunk {i} from URL {api_url}.")
            
            # Append the filtered OMM data to the list of chunks and count
            omm_data_chunks.append(json.dumps(filtered_omm_data))

            # Track unique NORAD IDs that returned OMMs for this chunk
            chunk_norad_ids_with_omms = {entry['NORAD_CAT_ID'] for entry in omm_data_json if 'NORAD_CAT_ID' in entry}

            # show logs for request
            print(f"Retrieved OMM data for chunk {i}: {len(chunk_string.split(','))} IDs requested of which {len(chunk_norad_ids_with_omms)} provided at least one OMM. "
                f"Total of {len(omm_data_json)} elements returned.")
            print(f"API URL: {api_url}")

            last_chunk_processed = i
            break

        # handle exceptions
        except ReadTimeout as e:
            retry_count += 1
            if retry_count > 1:
                print(f"ReadTimeout error for chunk {i}: {e}. Retry failed for URL: {api_url}")
                print("*** Checking each ID step by step ***")
                valid_data = check_ids_step_by_step(chunk_string)
                if valid_data:
                    omm_data_chunks.append(json.dumps(valid_data))
                print("Saving current state before proceeding to the next chunk...")
                dataframes = [pd.DataFrame(json.load(chunk)) for chunk in omm_data_chunks]                
                OMMs_df = pd.concat(dataframes, ignore_index=True)
                OMMs_df.to_csv('../Data/AllEpochs.csv', index=False)
                with open('../Data/last_chunk.txt', 'w') as f:
                    f.write(str(i))
                break  # Proceed to the next chunk
            else:
                print(f"ReadTimeout error for chunk {i}: {e}. Retrying in 10 seconds... (URL: {api_url})")
                time.sleep(10)

        except Exception as e:
            # 401 Unauthorized Error
            if '401' in str(e):
                print(f"Error 401 for chunk {i}: {e}. Attempting to re-login...")
                re_login()  # Call the re-login function

            # Other errors
            else:
                # Save the current state before raising the exception
                print(f"Error retrieving OMM data for chunk {i}: {e} (URL: {api_url})")
                print("Saving current state before exiting...")
                dataframes = [pd.DataFrame(json.loads(chunk)) for chunk in omm_data_chunks]                
                OMMs_df = pd.concat(dataframes, ignore_index=True)
                OMMs_df.to_csv('../Data/AllEpochs.csv', index=False)
                with open('../Data/last_chunk.txt', 'w') as f:
                    f.write(str(i))
                error_occurred = True
                break
    
    if error_occurred:
        break

    # Rate limit enforcement:  
    if request_counter % 25 == 0:
        print("Pausing for 60 seconds to avoid hitting the request limit...")
        time.sleep(60)

    # Add a delay of 5 seconds between API requests to avoid rate limiting
    if i < len(chunk_strings) - 1:
        print("Waiting for 5 seconds before the next request...")
        time.sleep(5)

# Final statistics
print(f"Processed chunks in this run: {start_chunk} to {last_chunk_processed} \n")

print("Final statistics:")
total_objects_requested = len(norad_ids)
total_objects_with_omms = OMMs_df['NORAD_CAT_ID'].nunique()
missing_objects = total_objects_requested - total_objects_with_omms
omm_data_received = len(OMMs_df)

print(f"Total OMMs requested: {total_objects_requested}")
print(f"Total objects that returned at least one OMM: {total_objects_with_omms} -> Missing: {missing_objects}")
print(f"Total OMM data elements received: {omm_data_received}")

# Save the OMM data chunks to a dataframe
dataframes = [pd.DataFrame(json.loads(chunk)) for chunk in omm_data_chunks]
OMMs_df = pd.concat(dataframes, ignore_index=True)

# Save the filtered dataframe to a CSV file
OMMs_df.to_csv('../Data/AllEpochs.csv', index=False)

Retrieved OMM data for chunk 0: 500 IDs requested of which 474 provided at least one OMM. Total of 35218 elements returned.
API URL: https://www.space-track.org/basicspacedata/query/class/gp_history/NORAD_CAT_ID/1,7,30,33,35,37,38,39,40,41,42,44,56,66,69,72,73,76,78,79,88,89,90,92,93,96,97,104,105,106,109,140,142,143,146,151,157,164,169,171,174,184,185,187,190,191,193,198,199,200,203,207,209,212,216,218,219,220,231,236,237,238,239,241,248,249,251,257,258,259,267,270,283,284,288,293,294,296,298,306,310,314,321,322,328,332,339,342,347,352,358,359,362,364,366,368,372,373,376,379,380,382,383,384,386,387,390,391,392,393,394,395,423,425,428,429,430,431,434,435,438,442,448,449,452,454,456,457,458,459,460,473,482,483,484,485,486,487,488,489,491,492,493,494,495,496,497,498,499,500,501,512,514,518,520,522,524,526,528,529,531,535,555,557,563,565,568,570,572,577,581,582,583,584,585,587,588,592,593,596,611,615,616,617,619,620,623,627,633,639,640,642,646,651,654,663,664,666,667,676,678,679,680,684,6

KeyboardInterrupt: 

In [52]:
# Save the OMM data chunks to a dataframe
dataframes = [pd.DataFrame(json.loads(chunk)) for chunk in omm_data_chunks]
OMMs_df = pd.concat(dataframes, ignore_index=True)

# show all columns of the df
pd.set_option('display.max_columns', None)
OMMs_df

,CCSDS_OMM_VERS,COMMENT,CREATION_DATE,ORIGINATOR,OBJECT_NAME,OBJECT_ID,CENTER_NAME,REF_FRAME,TIME_SYSTEM,MEAN_ELEMENT_THEORY,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,SEMIMAJOR_AXIS,PERIOD,APOAPSIS,PERIAPSIS,OBJECT_TYPE,RCS_SIZE,COUNTRY_CODE,LAUNCH_DATE,SITE,DECAY_DATE,FILE,GP_ID,TLE_LINE0,TLE_LINE1,TLE_LINE2
0,2.0,GENERATED VIA SPACE-TRACK.ORG API,2004-08-15T23:35:23,18 SPCS,TBA - TO BE ASSIGNED,1960-003A,EARTH,TEME,UTC,SGP4,1960-01-04T13:31:56.639135,15.12880250,0.03460000,51.2199,277.1069,269.5999,92.3479,0,U,30,999,1,0.00000000000000,0.00007271,0.0000000000000,6905.562,95.182,766.360,288.495,TBA,None,None,None,None,None,33906,8671893,0 TBA - TO BE ASSIGNED,1 00030U 60003 A 60004.56384999 .00007271 +0...,2 00030 051.2199 277.1069 0346000 269.5999 092...
1,2.0,GENERATED VIA SPACE-TRACK.ORG API,2004-08-15T23:35:23,18 SPCS,TBA - TO BE ASSIGNED,1960-003A,EARTH,TEME,UTC,SGP4,1960-01-05T03:48:47.663135,15.11508509,0.03516830,51.2199,274.2948,271.7583,90.2220,0,U,30,999,10,0.00000000000000,0.00006906,0.0000000000000,6909.740,95.269,774.609,288.601,TBA,None,None,None,None,None,33906,8671894,0 TBA - TO BE ASSIGNED,1 00030U 60003 A 60005.15888499 .00006906 +0...,2 00030 051.2199 274.2948 0351683 271.7583 090...
2,2.0,GENERATED VIA SPACE-TRACK.ORG API,2004-08-15T23:35:23,18 SPCS,TBA - TO BE ASSIGNED,1960-003A,EARTH,TEME,UTC,SGP4,1960-01-06T11:33:37.151999,15.11113949,0.03534990,51.2199,268.0439,276.5559,85.4254,0,U,30,999,30,0.00000000000000,0.00006901,0.0000000000000,6910.942,95.293,777.109,288.506,TBA,None,None,None,None,None,33906,8671895,0 TBA - TO BE ASSIGNED,1 00030U 60003 A 60006.48168000 .00006901 +0...,2 00030 051.2199 268.0439 0353499 276.5559 085...
3,2.0,GENERATED VIA SPACE-TRACK.ORG API,2004-08-15T23:35:23,18 SPCS,TBA - TO BE ASSIGNED,1960-003A,EARTH,TEME,UTC,SGP4,1960-01-11T02:38:37.317696,15.11556314,0.03515480,51.2199,246.1845,293.3337,68.4964,0,U,30,999,100,0.00000000000000,0.00042356,0.0000000000000,6909.594,95.266,774.364,288.554,TBA,None,None,None,None,None,33906,8671896,0 TBA - TO BE ASSIGNED,1 00030U 60003 A 60011.11015414 .00042356 +0...,2 00030 051.2199 246.1845 0351548 293.3337 068...
4,2.0,GENERATED VIA SPACE-TRACK.ORG API,2004-08-15T23:35:23,18 SPCS,TBA - TO BE ASSIGNED,1960-003A,EARTH,TEME,UTC,SGP4,1960-01-16T09:33:20.160000,15.12056877,0.03073890,51.2899,221.3629,308.2569,53.1149,0,U,30,999,180,0.00000000000000,0.00042245,0.0000000000000,6908.069,95.234,742.280,317.588,TBA,None,None,None,None,None,33906,8671897,0 TBA - TO BE ASSIGNED,1 00030U 60003 A 60016.39815000 .00042245 +0...,2 00030 051.2899 221.3629 0307389 308.2569 053...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
453582,2.0,GENERATED VIA SPACE-TRACK.ORG API,2004-05-28T18:53:53,18 SPCS,PSLV DEB,2001-049KJ,EARTH,TEME,UTC,SGP4,2002-01-23T08:02:20.930207,15.98629594,0.00439580,98.7118,107.4863,293.4859,65.5468,0,U,27312,999,193,0.03232000000000,0.10014836,0.3540600000000,6656.361,90.077,307.486,248.966,DEBRIS,MEDIUM,IND,2001-10-22,SRI,2002-01-23,6522,1995454,0 PSLV DEB,1 27312U 01049KJ 02023.33496447 +.10014836 +3...,2 27312 098.7118 107.4863 0043958 293.4859 065...
453583,2.0,GENERATED VIA SPACE-TRACK.ORG API,2004-05-28T18:53:56,18 SPCS,PSLV DEB,2001-049KJ,EARTH,TEME,UTC,SGP4,2002-01-23T08:20:11.260320,15.99057144,0.00440270,98.7114,107.5016,291.5841,138.7040,0,U,27312,999,194,0.03233100000000,0.09679410,0.3874900000000,6655.174,90.053,306.340,247.739,DEBRIS,MEDIUM,IND,2001-10-22,SRI,2002-01-23,6523,2002526,0 PSLV DEB,1 27312U 01049KJ 02023.34735255 +.09679410 +3...,2 27312 098.7114 107.5016 0044027 291.5841 138...
453584,2.0,GENERATED VIA SPACE-TRACK.ORG API,2004-08-17T15:29:51,18 SPCS,PSLV DEB,2001-049KK,EARTH,TEME,UTC,SGP4,2002-01-23T11:41:07.933631,15.81179757,0.005079

#### 2. Inspect data to see which columns are unneccecary 

In [7]:
# inspect certain columns to see if they can be dropped
print(OMMs_df['ORIGINATOR'].unique())
print(OMMs_df['CENTER_NAME'].unique())
print(OMMs_df['REF_FRAME'].unique())
print(OMMs_df['TIME_SYSTEM'].unique())
print(OMMs_df['MEAN_ELEMENT_THEORY'].unique())
print(OMMs_df['EPHEMERIS_TYPE'].unique())
print(OMMs_df['CLASSIFICATION_TYPE'].unique())
print(OMMs_df['ELEMENT_SET_NO'].unique())

['18 SPCS']
['EARTH']
['TEME']
['UTC']
['SGP4']
['0']
['U']
['999']


In [8]:
# drop not needed columns
OMMs_df.drop(columns=['CCSDS_OMM_VERS', 'COMMENT', 'CREATION_DATE', 'ORIGINATOR', 'CENTER_NAME', 'REF_FRAME', 'TIME_SYSTEM', 'MEAN_ELEMENT_THEORY', 'EPHEMERIS_TYPE', 'CLASSIFICATION_TYPE', 'ELEMENT_SET_NO', 'FILE', 'GP_ID', 'TLE_LINE0', 'TLE_LINE1', 'TLE_LINE2', 'COUNTRY_CODE'], inplace=True)
OMMs_df

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,NORAD_CAT_ID,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,SEMIMAJOR_AXIS,PERIOD,APOAPSIS,PERIAPSIS,OBJECT_TYPE,RCS_SIZE,LAUNCH_DATE,SITE,DECAY_DATE
0,THOR ABLESTAR R/B,1960-003A,1961-08-16T14:24:53.774496,16.29358349,0.00000000,51.2500,356.4892,331.9473,28.0526,30,7590,0.00000000000000,0.02125614,0.0000000000000,6572.405,88.378,194.270,194.270,ROCKET BODY,None,None,None,1961-08-18
1,THOR ABLESTAR DEB,1960-003C,1960-10-06T07:59:06.496512,15.38343354,0.02427570,51.2879,136.0668,196.7800,163.6100,33,2700,0.00000000000000,0.00001932,0.0000000000000,6829.149,93.607,616.796,285.231,DEBRIS,None,None,None,1960-07-17
2,SL-3 R/B,1960-005B,1960-07-16T04:00:11.150208,16.27523673,0.00000000,64.8899,44.6976,36.3819,323.6180,35,990,0.00000000000000,0.03271057,0.0000000000000,6577.344,88.477,199.209,199.209,ROCKET BODY,None,None,None,1960-07-17
3,SPUTNIK 4 DEB,1960-005D,1961-03-15T15:48:42.802271,15.49800338,0.01984030,64.8899,343.2411,333.4201,27.0840,37,4690,0.00000000000000,0.00039007,0.0000000000000,6795.450,92.915,552.139,282.492,DEBRIS,None,None,None,1961-06-30
4,SPUTNIK 4 DEB,1960-005E,1960-08-19T16:34:33.722976,15.88909013,0.00342130,64.8899,302.1218,49.4823,310.6691,38,1500,0.00000000000000,0.00610805,0.0000000000000,6683.481,90.628,328.212,282.480,DEBRIS,None,None,None,1960-08-20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24678,FALCON 9 R/B,2025-022Y,2025-02-19T02:13:45.786720,16.47531719,0.00310630,53.1436,118.6486,5.7660,67.2814,62878,274,0.00011008000000,0.10554311,0.0000130660000,6523.980,87.403,166.111,125.580,ROCKET BODY,LARGE,2025-02-01,AFWTR,2025-02-19
24679,VOLGA R/B,2025-026D,2025-02-05T11:52:04.781856,14.93887108,0.00100970,81.9945,220.5351,276.0393,83.9657,62905,3,0.00656530000000,0.00070488,0.0000000000000,6963.966,96.393,592.863,578.800,ROCKET BODY,None,None,None,2025-02-05
24680,ELECTRON R/B,2025-028F,2025-02-20T21:10:11.146080,16.39810484,0.00545400,97.9272,109.5562,197.5517,275.2945,62933,187,0.00027058000000,0.13714693,0.0000021991000,6544.444,87.815,202.002,130.615,ROCKET BODY,MEDIUM,2025-02-08,RLLC,2025-02-21
24681,ELECTRON R/B,2025-033C,2025-02-26T00:20:35.861280,16.43070160,0.00529660,58.9964,75.2812,270.2695,186.1265,63028,112,0.00030531000000,0.25107025,0.0000108760000,6535.785,87.641,192.268,123.033,ROCKET BODY,MEDIUM,2025-02-18,RLLC,2025-02-26


In [9]:
# save to csv
OMMs_df.to_csv("../Data/OMMs.csv", index=False)

#### 3. Check the differences between Decay Date and Epoch (timestamp of OMM)

In [10]:
# convert to date and then again to datetime, needed because otherwise the date conversion does not work :/
OMMs_df['EPOCH'] = pd.to_datetime(OMMs_df['EPOCH']).dt.date
OMMs_df['DECAY_DATE'] = pd.to_datetime(OMMs_df['DECAY_DATE']).dt.date

OMMs_df['EPOCH'] = pd.to_datetime(OMMs_df['EPOCH'])
OMMs_df['DECAY_DATE'] = pd.to_datetime(OMMs_df['DECAY_DATE'])

In [11]:
# check difference between Epoch and Decay Date
OMMs_df['Epoch_Decay_Difference'] = (OMMs_df['DECAY_DATE']-OMMs_df['EPOCH'])
OMMs_df['Epoch_Decay_Difference'].describe()

count                          24683
mean      19 days 05:21:06.118381072
std      300 days 09:36:12.104763172
min            -10727 days +00:00:00
25%                  0 days 00:00:00
50%                  1 days 00:00:00
75%                  5 days 00:00:00
max               8300 days 00:00:00
Name: Epoch_Decay_Difference, dtype: object

In [12]:
# show objects where Epoch_Decay_Difference is negative
OMMs_wrong_df = OMMs_df[OMMs_df['Epoch_Decay_Difference'] < pd.Timedelta(0)]
OMMs_wrong_df

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,NORAD_CAT_ID,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,SEMIMAJOR_AXIS,PERIOD,APOAPSIS,PERIAPSIS,OBJECT_TYPE,RCS_SIZE,LAUNCH_DATE,SITE,DECAY_DATE,Epoch_Decay_Difference
1,THOR ABLESTAR DEB,1960-003C,1960-10-06,15.38343354,0.02427570,51.2879,136.0668,196.7800,163.6100,33,2700,0.00000000000000,0.00001932,0.0000000000000,6829.149,93.607,616.796,285.231,DEBRIS,None,None,None,1960-07-17,-81 days
2872,DELTA 1 DEB,1972-058C,2004-11-24,15.86540389,0.00128020,98.3597,332.5522,114.4864,245.7741,7832,57786,0.00141360000000,0.00371465,0.0000000000000,6690.132,90.763,320.561,303.432,DEBRIS,LARGE,1972-07-23,AFWTR,1977-02-12,-10147 days
3087,COSMOS 777 DEB,1975-102F,1976-05-02,15.50607889,0.00266690,65.0265,340.8773,6.7572,353.3878,8625,2876,0.00354340000000,0.00212521,0.0000000000000,6793.091,92.866,433.072,396.839,DEBRIS,None,None,None,1976-03-15,-48 days
3100,COSMOS 777 DEB,1975-102AE,1976-05-02,15.50641491,0.00104770,65.0344,341.0843,8.9752,351.1527,8650,2877,0.00000000000000,0.00246035,0.0000000000000,6792.993,92.864,421.975,407.741,DEBRIS,None,None,None,1976-03-27,-36 days
3142,COSMOS 777 DEB,1975-102BF,1976-05-02,15.40156937,0.00557380,64.8281,339.9362,23.9313,336.4347,8728,2872,0.00328330000000,0.00142121,0.0000000000000,6823.787,93.496,483.686,407.617,DEBRIS,None,None,None,1976-03-29,-34 days
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23254,BION 11 DEB,1996-073G,1997-02-03,16.25788645,0.00087860,62.7868,87.1680,23.0743,337.0808,24710,439,0.00004231000000,0.00130341,0.0000086059000,6582.022,88.572,209.670,198.105,DEBRIS,SMALL,1996-12-24,PKMTR,1997-01-10,-24 days
23897,CZ-2F DEB,2024-078D,2024-04-28,16.31592083,0.00635030,41.5973,28.4800,152.0982,294.6586,59594,37,0.00245460000000,0.38721847,0.0000072850000,6566.402,88.257,229.965,146.568,DEBRIS,None,2024-04-25,JSC,2024-04-27,-1 days
24120,COSMOS 2251 DEB,1993-036AZQ,2024-05-11,14.67237796,0.00346710,73.9910,229.2548,89.3171,271.1971,36387,57450,0.02934800000000,0.00171058,0.0000000000000,7048.037,98.144,694.338,645.466,DEBRIS,SMALL,1993-06-16,PKMTR,2024-05-07,-4 days
24312,COSMOS 1408 DEB,1982-092CCH,2023-07-06,15.17775450,0.01267330,82.5778,267.2573,149.0686,211.8095,57287,1,0.00109990000000,0.00027530,0.0000000000000,6890.702,94.876,599.895,425.239,DEBRIS,None,None,None,2023-06-12,-24 days


In [13]:
# save IDs of these objects to csv to later also delete from SATCAT data
wrong_ids = OMMs_wrong_df['NORAD_CAT_ID']
wrong_ids.to_csv("../Data/WrongIDs.csv", index=False)

In [14]:
# drop objects where OMM recorded after object was supposed to have decayed
OMMs_df = OMMs_df[OMMs_df['Epoch_Decay_Difference'] >= pd.Timedelta(0)]
print(f"Remaining objects: {len(OMMs_df)}")



Remaining objects: 24257


In [15]:
# Desribe in days
OMMs_df['Epoch_Decay_Difference'] = OMMs_df['Epoch_Decay_Difference'].dt.days
OMMs_df['Epoch_Decay_Difference'].describe()

/var/folders/xm/n65_n9sd78z4_zdw8r87klf00000gn/T/ipykernel_4716/1006036595.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  OMMs_df['Epoch_Decay_Difference'] = OMMs_df['Epoch_Decay_Difference'].dt.days


count    24257.000000
mean        34.496805
std        222.906075
min          0.000000
25%          0.000000
50%          1.000000
75%          5.000000
max       8300.000000
Name: Epoch_Decay_Difference, dtype: float64

In [16]:
# remove the difference and epoch column 
OMMs_df.drop(columns=['EPOCH', 'Epoch_Decay_Difference'], inplace=True)

/var/folders/xm/n65_n9sd78z4_zdw8r87klf00000gn/T/ipykernel_4716/1425059375.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  OMMs_df.drop(columns=['EPOCH', 'Epoch_Decay_Difference'], inplace=True)


In [17]:
# Save OMMs to csv
OMMs_df.to_csv('../Data/OMMs.csv', index=False)